<a href="https://colab.research.google.com/github/headdown0845/2026_First-semester/blob/main/2355022_%EC%9D%B4%ED%98%95%EB%AF%BC_4%EC%9B%94_%EB%8D%B0%EC%9D%B4%ED%84%B0%EA%B4%80%EB%A6%AC%EB%A1%A0_%EA%B3%BC%EC%A0%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [13]:
# [Step 1] 데이터 로드 및 샘플링
categories = ['comp.graphics', 'sci.space', 'talk.religion.misc']
newsgroups = fetch_20newsgroups(subset='train', categories=categories,
                                remove=('headers', 'footers', 'quotes'))

In [14]:
data, labels = [], []
for i, category in enumerate(categories):
    indices = np.where(newsgroups.target == i)[0][:20]  # 각 주제별 20개 추출
    for idx in indices:
        data.append(newsgroups.data[idx])
        labels.append(newsgroups.target_names[i])


In [17]:

# [Step 2] CountVectorizer 설정 및 변환
# TODO: CountVectorizer를 선언하고 데이터를 fit_transform 하세요.
vectorizer = CountVectorizer(stop_words='english')
count_matrix = vectorizer.fit_transform(data)

In [18]:
# [Step 3] 분류 함수 구현
def classify_text(input_text):
    # TODO: 입력 텍스트를 벡터화하고 코사인 유사도를 계산하는 로직을 작성하세요.
    input_vec = vectorizer.transform([input_text])
    sim = cosine_similarity(input_vec, count_matrix)

    best_idx = np.argmax(sim)
    return labels[best_idx], sim[0][best_idx]

In [19]:
# [Step 4] 테스트 실행
test_sentences = [
    "The rocket launched into orbit.",
    "A new 3D rendering technique for graphics.",
    "Theological discussions on faith and god."
]

for s in test_sentences:
    cat, score = classify_text(s)
    print(f"문장: {s[:30]}... | 예측: {cat} | 유사도: {score:.4f}")

문장: The rocket launched into orbit... | 예측: sci.space | 유사도: 0.0870
문장: A new 3D rendering technique f... | 예측: comp.graphics | 유사도: 0.1952
문장: Theological discussions on fai... | 예측: talk.religion.misc | 유사도: 0.3430


Q1. 유사도가 0.0000이 나오는 이유 분석 특정 문장(예: "Exploring the mars with a robotic
rover.")을 입력했을 때 유사도가 0이 나오거나 엉뚱한 카테고리가 매칭되는 이유를
CountVectorizer의 동작 원리와 관련지어 설명하세요.

In [23]:
#Q1
test_sentences = [
    "Exploring the mars with a robotic rover."
]

for s in test_sentences:
    cat, score = classify_text(s)
    print(f"문장: {s[:30]}... | 예측: {cat} | 유사도: {score:.4f}")

문장: Exploring the mars with a robo... | 예측: comp.graphics | 유사도: 0.0000


A1. CountVectorizer는 학습 데이터에 등장한 단어들로만 단어 사전을 만들기에  "Exploring the mars with a robotic rover." 와 같은 문장을 입력 했을 때 유사도가 0이 나온다면 기존에 수집한 샘플 데이터에 해당 문장을 구성하는 단어가 등장하지 않았기 때문입니다.
또한, CountVectorizer는 문맥을 보지 않고 단어의 출현 빈도만 따지기 때문에 엉뚱한 카테고리가 매칭될 수 있습니다.

Q2. 성능 개선 실험 유사도 점수를 높이고 분류 정확도를 올리기 위해 다음 중 하나를 시도하고
결과를 비교하세요.

● 주제별 샘플 수를 20개에서 100개로 늘렸을 때의 변화.

● CountVectorizer 대신 TfidfVectorizer를 사용했을 때의 차이점.

● ngram_range=(1, 2) 설정을 추가했을 때의 영향.

A2. ● ngram_range=(1, 2) 설정을 추가했을 때의 영향.

ngram_range는 단어를 몇 개씩 묶는 것입니다.

만약 ngram_range = (1, 2)라고 한다면, 단어의 묶음을 1개부터 2개까지 설정하라는 뜻입니다. 기존에는 'data', 'science' 각각 분류 했다면 적용 후에는 'data science'로 분류가 가능해집니다.
이로 인한 영향은 문맥과 의미 파악 능력이 향상 되지만 Feature 개수가 급격히 증가하여 계산 속도가 느려질 수도 있습니다.

 유사도가 더 높아져야 하지만 아래에서 더 낮아진 이유는 데이터셋의 크기가 작기에 오히려 희소성 문제로 인해 유사도가 낮아지는 결과가 나왔습니다.

In [24]:
vectorizer = CountVectorizer(stop_words='english', ngram_range=(1, 2))
count_matrix = vectorizer.fit_transform(data)

In [25]:
def classify_text(input_text):

    input_vec = vectorizer.transform([input_text])
    sim = cosine_similarity(input_vec, count_matrix)

    best_idx = np.argmax(sim)
    return labels[best_idx], sim[0][best_idx]

In [26]:
test_sentences = [
    "The rocket launched into orbit.",
    "A new 3D rendering technique for graphics.",
    "Theological discussions on faith and god."
]

for s in test_sentences:
    cat, score = classify_text(s)
    print(f"문장: {s[:30]}... | 예측: {cat} | 유사도: {score:.4f}")

문장: The rocket launched into orbit... | 예측: sci.space | 유사도: 0.0685
문장: A new 3D rendering technique f... | 예측: comp.graphics | 유사도: 0.1469
문장: Theological discussions on fai... | 예측: talk.religion.misc | 유사도: 0.2462
